## Querying GeoSpatial data on US Census using DuckDB



In [ ]:
import duckdb


In [ ]:
# Connect to DuckDB and load the spatial extension
con = duckdb.connect()
con.execute("INSTALL spatial; LOAD spatial;") ;


In [ ]:
roads_url = "https://www2.census.gov/geo/tiger/TIGER2024/ROADS/tl_2024_35006_roads.zip"
states_url = "https://www2.census.gov/geo/tiger/TIGER2024/STATE/tl_2024_us_state.zip"

roads_path = f"/vsizip//vsicurl/{roads_url}/tl_2024_35006_roads.shp"
states_path = f"/vsizip//vsicurl/{states_url}/tl_2024_us_state.shp"


In [ ]:
! curl -s -I {roads_url}


In [ ]:
! curl -s -I {states_url}


In [ ]:
query = f"""
    SELECT
        roads.* EXCLUDE (geom),
        states.STATEFP,
        states.STUSPS,
        states.NAME AS state_name,
        roads.geom AS road_geometry,
        states.geom AS state_geometry
    FROM st_read('{roads_path}') roads
    LEFT JOIN st_read('{states_path}') states
        ON ST_Within(roads.geom, states.geom)
    WHERE roads.FULLNAME LIKE '%I- 25%'
       OR (roads.RTTYP = 'I'
       -- AND roads.FULLNAME LIKE '%25%'
       )
"""
print(query)


In [ ]:
gdf_combined = con.sql(query).df()
gdf_combined


In [ ]:
query = f"""
  SELECT count(1)
  FROM st_read('{roads_path}')
"""

con.execute(query)
